# Run radcoolpv in Google Colab

This notebook is for students who want to edit a YAML experiment and run radcoolpv without installing Python or S4 on their own computer.

**Prerequisites:** basic YAML syntax and the optical/thermal conventions summarized on the teaching site. No local software installation is required.

**Learning goals**

1. Build the native S4 Python extension in a temporary Colab runtime.
2. Edit and validate a YAML configuration.
3. Obtain reflectance, transmittance, absorptance/emittance, and an energy-closure check.
4. Obtain operating temperature, current-voltage behavior, maximum power, fill factor, and efficiency from a stored spectrum.

> The setup and S4 compilation take several minutes in a new runtime. Colab runtimes are temporary, so repeat the setup after a reset. The default calculation is a smoke test, not a converged scientific result.

**Outline:** prepare the runtime; edit and validate YAML; obtain optical properties; obtain PV parameters and figures; review the validation evidence; examine the full coupled configuration.


## Why import S4 does not work before setup

S4 exposes a Python interface, but it is not a pure-Python file. Its C and C++ sources must be compiled into a native extension matching the operating system, processor architecture, and active CPython version. Python can import that binary only after it has been built and installed into the current environment.

radcoolpv imports S4 lazily: thermal calculations and stored-spectrum readers work without S4, while geometry.source: s4 requires the compiled extension. Colab gives us a Linux machine on which to build it.


## 1. Prepare the Colab runtime

Run the next three cells once per new runtime. They install Linux build libraries, clone this repository, and compile a pinned S4 revision.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")
S4_DIR = Path("/content/S4")

def run_command(args: list[str], cwd: Path | None = None) -> None:
    print("$", " ".join(args))
    subprocess.run(args, cwd=cwd, check=True)


In [ ]:
run_command(["apt-get", "-qq", "update"])
run_command([
    "apt-get", "-qq", "install", "-y",
    "build-essential", "git", "libboost-all-dev", "libfftw3-dev",
    "liblapack-dev", "libopenblas-dev", "libsuitesparse-dev",
])


In [ ]:
if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)


In [ ]:
# Pinned for reproducibility; this is the phoebe-p/S4 devel revision from 2025-06-11.
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"

if importlib.util.find_spec("S4") is None:
    if not S4_DIR.exists():
        run_command(["git", "clone", "https://github.com/phoebe-p/S4.git", str(S4_DIR)])
    run_command(["git", "checkout", S4_COMMIT], cwd=S4_DIR)
    run_command(["make", "-j2", "S4_pyext"], cwd=S4_DIR)
    importlib.invalidate_caches()

import S4
import radcoolpv

print("S4:", S4.__file__)
print("radcoolpv:", radcoolpv.__file__)


## 2. Edit the YAML experiment

Change values in the next cell and run it again to rewrite student.yaml. The small grid and s4_modes: 30 are intentionally cheap. They do not establish numerical convergence.


In [ ]:
%%writefile student.yaml
# Teaching smoke test only. Do not report these settings as converged.
run:
  optics: true
  thermal: false
  plots: true
  mode: standard
  results_dir: results
  write_outputs: true

simulation:
  wavelength: {min: 8.0, max: 13.0, n: 21}
  angles: normal
  polarization: unpolarized
  s4_modes: 30

geometry:
  source: s4
  shape: cylinder
  photonic_material: sio2
  lattice: {type: square, x: 20.0, y: 20.0}
  discretization_layers: 1
  cylinder: {radius: 5.0, height: 30.0}

structure:
  - {material: sio2, thickness: 100.0}
  - {material: si3n4, thickness: 0.075}
  - {material: silicon, thickness: 250.0}
  - {material: substrate, thickness: 0.0, terminal: true}

materials:
  sio2: PalikKitamura_SiO2
  silicon: SiliconNew
  si3n4: DrudeSi3N4
  substrate: Hagemann_Ag


Validate first. --print-config parses the YAML and shows the resolved settings without running S4.


In [ ]:
run_command([sys.executable, "-m", "radcoolpv.cli", "run", "student.yaml", "--print-config"])


## 3. Run and inspect the result

The next cell executes live S4 optics. Runtime depends on the Colab machine and your YAML settings.


In [ ]:
run_command([sys.executable, "-m", "radcoolpv.cli", "run", "student.yaml"])


In [ ]:
from IPython.display import Image, Markdown, display
import numpy as np

manifests = list((PROJECT_DIR / "results").glob("*/run.json"))
latest_manifest = max(manifests, key=lambda path: path.stat().st_mtime)
record = json.loads(latest_manifest.read_text())

optics = np.genfromtxt(latest_manifest.parent / "optics.csv", delimiter=",", names=True)
closure_error = np.max(np.abs(optics["ref"] + optics["tran"] + optics["emit"] - 1.0))
window_mean = np.trapz(optics["emit"], optics["lambda_um"]) / 5.0

display(Markdown(
    "| Optical result | Value |\n|---|---:|\n"
    f"| Wavelength samples | {len(optics)} |\n"
    f"| max $\\lvert R+T+A-1 \\rvert$ | {closure_error:.3e} |\n"
    f"| Wavelength-averaged 8–13 µm emittance | {window_mean:.4f} |"
))
print("Results folder:", latest_manifest.parent)
print("S4 provenance:", json.dumps(record["provenance"].get("s4"), indent=2))

for figure in sorted((latest_manifest.parent / "figures").glob("*.png")):
    display(Image(filename=str(figure)))


## 4. Obtain thermal and PV parameters

A complete hemispherical S4 calculation is too expensive for a short classroom run. The next YAML therefore resumes the committed spectrum from Validation A.1 and executes the thermal/electrical model. This produces the main PV parameters and figures without recalculating optics.

> **Evidence limit:** this is an end-to-end software example, not an independent literature validation. The underlying geometry assumes a close-packed pitch, and the stored spectrum is TE at normal incidence. The thermal model treats that spectrum as angle-independent.


In [ ]:
%%writefile student_pv.yaml
# Fast PV tutorial based on the committed Validation A.1 spectrum.
# This case is not an independent literature validation.
run:
  optics: false
  thermal: true
  plots: true
  mode: standard
  results_dir: tutorial-results/pv
  write_outputs: true
  optics_results: validations/validation A.1/data/optics/hemisph_sodalime_te_normal.txt
  optics_results_angles: normal

simulation:
  wavelength: {min: 0.3, max: 30.0, n: 2000}
  angles: normal

structure:
  - {material: silicon, thickness: 250.0}
  - {material: substrate, thickness: 0.0, terminal: true}

thermal:
  ambient_temperature: 298
  convection_coefficient: 12
  voltage: {min: 0.1, max: 0.8, n: 100}
  equilibrium: auto
  pv:
    series_resistance: 0.00011
    shunt_resistance: 0.1
    bandgap: {eg0: 1.166, alpha: 4.73e-4, beta: 636}
    iqe_file: data/siliconIQE.txt

data:
  solar_spectrum: data/astmg173.xlsx
  atmosphere: data/atmosphericData/cptrans_nq_100_15.dat


Validate the resolved paths and parameters, then run the PV case. Edit one YAML parameter at a time so that changes in the result remain interpretable.


In [ ]:
run_command([sys.executable, "-m", "radcoolpv.cli", "run", "student_pv.yaml", "--print-config"])
run_command([sys.executable, "-m", "radcoolpv.cli", "run", "student_pv.yaml"])


In [ ]:
pv_manifests = list((PROJECT_DIR / "tutorial-results" / "pv").glob("PV-results_*/run.json"))
latest_pv_manifest = max(pv_manifests, key=lambda path: path.stat().st_mtime)
pv_record = json.loads(latest_pv_manifest.read_text())
thermal = pv_record["thermal_results"]
averages = pv_record["band_averages_percent"]

pv_rows = [
    ("Equilibrium temperature", thermal["equilibrium_temperature_K"], "K"),
    ("Short-circuit current density", thermal["short_circuit_current_A_per_m2"], "A/m²"),
    ("Open-circuit voltage at equilibrium", thermal["voc_equilibrium_V"], "V"),
    ("Voltage at maximum power", thermal["vmpp_V"], "V"),
    ("Maximum power at equilibrium", thermal["mpp_equilibrium_W_per_m2"], "W/m²"),
    ("Fill factor at equilibrium", 100.0 * thermal["fill_factor_equilibrium"], "%"),
    ("Efficiency at equilibrium", 100.0 * thermal["efficiency_equilibrium"], "%"),
    ("Solar-weighted silicon absorptance", averages["solar_absorptance_silicon"], "%"),
    ("Blackbody-weighted 8–13 µm emittance", averages["emittance_8_13um"], "%"),
]
table = ["| PV result | Value | Unit |", "|---|---:|---|"]
table.extend(f"| {name} | {value:.4g} | {unit} |" for name, value, unit in pv_rows)
display(Markdown("\n".join(table)))
print("Results folder:", latest_pv_manifest.parent)

for figure in sorted((latest_pv_manifest.parent / "figures").glob("*.png")):
    display(Image(filename=str(figure)))


### Interpret the outputs

- `optical_properties.png` separates total absorptance/emittance from absorption in silicon. Only silicon absorption generates photocurrent in this model.
- `energy_balance_terms.png` shows the terms whose zero determines the equilibrium temperature.
- `iv_curve.png` and `power_curve.png` show the electrical operating point; maximum power is not located at either $I_{sc}$ or $V_{oc}$.
- `efficiency.png` shows the temperature dependence of conversion efficiency over the solved thermal grid.

For the committed A.1 spectrum, the reference values are 317.09 K, 365.92 A/m², 0.7264 V equilibrium $V_{oc}$, 230.61 W/m² equilibrium maximum power, fill factor 0.868, and efficiency 23.14%. Small last-digit differences can arise from numerical-library versions. Larger differences require investigation.


## 5. Full coupled configuration

The repository's configs/full.yaml enables live hemispherical optics and the thermal/PV balance. Its default grid has 2000 wavelengths, 97 directions including the normal probe, and two polarizations: about 388000 S4 solves. That is generally unsuitable for a free Colab session.

The next cell downloads the complete file so you can inspect and edit it. Start with a reduced grid, make the workflow run, and then perform a stated convergence study. Do not mistake a reduced classroom grid for a publishable result.


In [ ]:
from urllib.request import urlretrieve

FULL_CONFIG_URL = "https://raw.githubusercontent.com/gsilvaoelker/radcoolpv-py/main/configs/full.yaml"
full_config = PROJECT_DIR / "student_full.yaml"
urlretrieve(FULL_CONFIG_URL, full_config)
full_lines = full_config.read_text().splitlines()
print("\n".join(full_lines[:80]))


In [ ]:
# Deliberate safety gate: edit student_full.yaml and define a convergence plan first.
RUN_FULL_CASE = False

if RUN_FULL_CASE:
    run_command([sys.executable, "-m", "radcoolpv.cli", "run", str(full_config)])
else:
    print("Full coupled run skipped. Set RUN_FULL_CASE = True only when the grid is intentional.")


## 6. Use the supplied validations correctly

The repository includes conditional regressions, smoke tests, partial validation, and one documented thermal failure. They cannot be cited interchangeably. Read the teaching site's [Validation evidence](../validations.md) page before selecting a case, and retain each run's `run.json` with any result you report.


## Exercises

1. Change the cylinder radius while keeping the lattice fixed. Predict which spectral features may move before running the case.
2. Compare TE and TM at one nonzero polar angle using angles: specific.
3. Increase s4_modes systematically and plot one scalar observable against basis size.
4. Explain why agreement with $R+T+A=1$ is necessary but not sufficient evidence of convergence.
5. Change one PV parameter in student_pv.yaml and explain the direction of the resulting change before rerunning it.
6. Use the scaffold below to compare the solve count of the smoke test and full case.


In [ ]:
def estimated_s4_solves(
    n_lambda: int,
    n_polarizations: int,
    *,
    hemispherical: bool = False,
    n_theta: int = 1,
    n_phi: int = 1,
) -> int:
    directions = 1 + n_theta * n_phi if hemispherical else 1
    return n_lambda * directions * n_polarizations

smoke_test_solves = estimated_s4_solves(21, 2)
full_case_solves = estimated_s4_solves(
    2000, 2, hemispherical=True, n_theta=8, n_phi=12
)
smoke_test_solves, full_case_solves
